In [1]:
import torch

In [54]:
def check_pruned_weights(iteration):
    track_neurons = {}
    weights = torch.load(f"models/lottery_ticket/Run2Full_allneurons/{iteration}_Pruning_Iter/model_best.pth")['state_dict']
    for layer in weights.keys():
        if any(x in layer for x in ['embedding', 'bias', 'bn', 'LayerNorm']):
            continue
        ct, neurons = full_neurons_pruned(weights[layer])
        track_neurons[layer] = {'num_neurons_pruned_out': ct, 'neurons_pruned_out': neurons}
    return track_neurons
def full_neurons_pruned(w):
    w=w.t()
    ct=0
    ns=[]
    for neuron, weights in enumerate(w):
        if neuron == 308:
            print(weights.sum())
        if torch.where(weights==0,1,0).sum() == 0:
            ct += 1
            ns.append(neuron)
    return ct, ns
pruning_changes = {}
for iteration in range(2,3):
    tracked_neurons = check_pruned_weights(iteration)
    num_neurons_pruned_at_iter = torch.tensor([tracked_neurons[layer]['num_neurons_pruned_out'] for layer in tracked_neurons.keys()])
    neurons_pruned_at_iter =[tracked_neurons[layer]['neurons_pruned_out'] for layer in tracked_neurons.keys()]
    pruning_changes[iteration] = {'count': num_neurons_pruned_at_iter, 'neurons': neurons_pruned_at_iter}
layers = tracked_neurons.keys()

tensor(0.0032, device='cuda:0')
tensor(-1.1192, device='cuda:0')
tensor(-0.1294, device='cuda:0')
tensor(-0.1024, device='cuda:0')
tensor(28.7519, device='cuda:0')
tensor(0.2528, device='cuda:0')
tensor(1.5054, device='cuda:0')
tensor(0.5583, device='cuda:0')
tensor(0.2576, device='cuda:0')
tensor(0.0297, device='cuda:0')
tensor(47.0890, device='cuda:0')
tensor(0.1558, device='cuda:0')
tensor(-1.6479, device='cuda:0')
tensor(-2.2198, device='cuda:0')
tensor(-0.3900, device='cuda:0')
tensor(-0.1986, device='cuda:0')
tensor(63.6439, device='cuda:0')
tensor(-0.1654, device='cuda:0')
tensor(-0.1412, device='cuda:0')
tensor(0.2301, device='cuda:0')


KeyboardInterrupt: 

In [ ]:
tracked_neurons

In [46]:
import re
import collections
import pandas as pd
import os
def get_indiv_concepts(formula) -> list:
    concepts=[]
    concps = re.findall(r'\b(?:NOT )?(?:pre:tok:|pre:tag:|hyp:tag:|hyp:tok:|oth:)\S*', formula)
    for c in concps:
        try:
            end_idx =c.index(')')
        except:
            end_idx = len(c)
        concepts.append(c[:end_idx])
    return concepts
                
def get_collected_concepts(root):
    concepts_per_neuron_per_cluster_per_pruningIteration = collections.defaultdict(lambda : collections.defaultdict(lambda : collections.defaultdict(list)))
    #for each pruning iter
    neurons=collections.defaultdict(list)
    for pruning_iter in os.listdir(root):
        if '.ipynb' in pruning_iter: continue
        pruned_percent = pruning_iter[:pruning_iter.index("_")]
        #for each cluster
        for cluster in os.listdir(os.path.join(root, pruning_iter)):
            if cluster == 'result.csv' or '.ipy' in cluster: continue
  
            cluster_num = cluster[7]
            for row in pd.read_csv(os.path.join(root, pruning_iter, cluster)).iterrows(): #fix this
                row = row[1]
                neuron, formula, iou = row.unit, row.best_name,float("{:.3f}".format(row.best_iou))
                formula = get_indiv_concepts(formula)
                #print(formula)
                for concept in formula:
                    # If concept doesn't exist, initialize all nested levels
                    if concept not in concepts_per_neuron_per_cluster_per_pruningIteration:
                        concepts_per_neuron_per_cluster_per_pruningIteration[concept] = {}

                    prune_key = f"{pruned_percent}% Prune"
                    if prune_key not in concepts_per_neuron_per_cluster_per_pruningIteration[concept]:
                        concepts_per_neuron_per_cluster_per_pruningIteration[concept][prune_key] = {
                            f"Cluster{i}": [] for i in range(1,4)
                        }

                    cluster_key = f"Cluster{cluster_num}"
                    if cluster_key not in concepts_per_neuron_per_cluster_per_pruningIteration[concept][prune_key]:
                        concepts_per_neuron_per_cluster_per_pruningIteration[concept][prune_key][cluster_key] = []

                    # Now append the neuron info
                    neurons[neuron].append({'concept': concept, 'iou': iou, 'prune_iter':pruned_percent, 'cluster':cluster_key})
                    concepts_per_neuron_per_cluster_per_pruningIteration[concept][prune_key][cluster_key].append(f"Neuron {neuron}: IOU:{iou}")
    return neurons, concepts_per_neuron_per_cluster_per_pruningIteration
root='/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run2Full_allneurons/Expls'    
neurons,concepts=get_collected_concepts(root)

In [116]:
thresh=50
def per_layer_pruning_differences(iter_, layers, pruning_changes):
    weights = torch.load(f"models/lottery_ticket/Run3/{iter_}_Pruning_Iter/model_best.pth")['state_dict']
    for i,layer in enumerate(layers):
        pruned = 0 
        dead_neurons=[]
        if layer=='mlp.0.weight':
            print(weights[layer].shape)
        for neuron, connection in enumerate(weights[layer]:
            numnonzero = torch.where(connection > 0, 1,0).sum()
            if 0 < numnonzero < thresh:
                
                dead_neurons.append(neuron)
            if numnonzero == 0:
                #print(f"\t Layer {layer}: {neuron} pruned")
                pruned += 1
        print(f"At layer {layer}, ============{pruned}/{weights[layer].shape[0]}================ neurons pruned ")#at {SPARSITIES[iter_-1]}%")
ACCS= [ 0.858, 0.857, 0.859, 0.859, 0.858, 0.855, 0.856, 0.856,  0.853, 0.850,0.844, 0.836,0.830, 0.830, 0.826]
SPARSITIES = [0.2, 0.36, 0.4879999603599318, 0.5903999682879455, 0.6723199349902882, 0.7378560669124351, 0.7902848138898799, 0.832227870931938, 0.8657822967455504, 0.8926258572164744, 0.914100765053316, 0.9312805724025845, 0.9450244777421017, 0.9560195425536132]

In [117]:
per_layer_pruning_differences(7, layers, pruning_changes)

At layer encoder.encoder.layer.0.attention.self.query.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.self.key.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.self.value.weight, ============1/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.output.dense.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.intermediate.dense.weight, ============0/3072================ neurons pruned 
At layer encoder.encoder.layer.0.output.dense.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.query.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.key.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.value.weight, ============0/768================ neurons pruned 
At 

In [115]:
per_layer_pruning_differences(8, layers, pruning_changes)

At layer encoder.encoder.layer.0.attention.self.query.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.self.key.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.self.value.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.attention.output.dense.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.0.intermediate.dense.weight, ============0/3072================ neurons pruned 
At layer encoder.encoder.layer.0.output.dense.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.query.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.key.weight, ============0/768================ neurons pruned 
At layer encoder.encoder.layer.1.attention.self.value.weight, ============0/768================ neurons pruned 
At 

In [101]:
per_layer_pruning_differences(14, layers, pruning_changes)

At layer mlp.0.weight, dead neurons are [2, 4, 5, 8, 11, 22, 31, 32, 38, 45, 68, 71, 83, 98, 99, 100, 121, 124, 126, 136, 153, 157, 161, 167, 170, 180, 192, 209, 219, 233, 236, 241, 243, 252, 255, 259, 265, 272, 291, 293, 295, 298, 303, 305, 316, 329, 331, 334, 346, 348, 357, 360, 374, 378, 381, 382, 392, 399, 416, 422, 433, 441, 442, 447, 452, 457, 469, 471, 472, 473, 485, 494, 495, 515, 519, 528, 529, 533, 544, 549, 551, 576, 580, 585, 595, 596, 600, 603, 610, 625, 650, 651, 653, 656, 659, 669, 673, 685, 689, 714, 720, 722, 729, 735, 749, 770, 775, 777, 783, 784, 799, 803, 810, 814, 822, 823, 826, 833, 838, 841, 843, 847, 869, 898, 905, 910, 911, 914, 928, 939, 944, 946, 955, 961, 962, 973, 980, 987, 995, 999, 1001, 1009, 1011] and ============0/1024================ neurons pruned 


In [76]:
per_layer_pruning_differences(10, layers, pruning_changes)

At layer encoder.encoder.layer.0.attention.self.query.weight, ============0/768================ neurons pruned at 0.8926258572164744%
At layer encoder.encoder.layer.0.attention.self.key.weight, ============0/768================ neurons pruned at 0.8926258572164744%
	 3 connecs left for neuron 233
At layer encoder.encoder.layer.0.attention.self.value.weight, ============0/768================ neurons pruned at 0.8926258572164744%
At layer encoder.encoder.layer.0.attention.output.dense.weight, ============0/768================ neurons pruned at 0.8926258572164744%
	 2 connecs left for neuron 639
	 4 connecs left for neuron 950
	 4 connecs left for neuron 1057
	 3 connecs left for neuron 1485
	 4 connecs left for neuron 1670
	 4 connecs left for neuron 1726
	 4 connecs left for neuron 2047
	 1 connecs left for neuron 2271
	 3 connecs left for neuron 2511
	 4 connecs left for neuron 2705
	 2 connecs left for neuron 2794
	 4 connecs left for neuron 2819
	 3 connecs left for neuron 2931
At la

In [77]:
per_layer_pruning_differences(14, layers, pruning_changes)

	 2 connecs left for neuron 14
	 3 connecs left for neuron 15
	 4 connecs left for neuron 21
	 4 connecs left for neuron 25
	 2 connecs left for neuron 26
	 2 connecs left for neuron 33
	 3 connecs left for neuron 36
	 4 connecs left for neuron 37
	 4 connecs left for neuron 38
	 4 connecs left for neuron 41
	 2 connecs left for neuron 54
	 Layer encoder.encoder.layer.0.attention.self.query.weight: 58 pruned
	 4 connecs left for neuron 59
	 3 connecs left for neuron 61
	 4 connecs left for neuron 63
	 4 connecs left for neuron 219
	 4 connecs left for neuron 236
	 3 connecs left for neuron 262
	 4 connecs left for neuron 270
	 3 connecs left for neuron 277
	 2 connecs left for neuron 305
	 4 connecs left for neuron 307
	 4 connecs left for neuron 308
	 3 connecs left for neuron 309
	 3 connecs left for neuron 313
	 1 connecs left for neuron 392
	 2 connecs left for neuron 393
	 4 connecs left for neuron 395
	 4 connecs left for neuron 396
	 2 connecs left for neuron 398
	 2 connecs lef

KeyboardInterrupt: 

In [63]:
per_layer_pruning_differences(14, layers, pruning_changes)

	 3 connecs left for neuron 233
	 4 connecs left for neuron 334
	 4 connecs left for neuron 351
	 4 connecs left for neuron 385
	 4 connecs left for neuron 395
	 3 connecs left for neuron 451
	 3 connecs left for neuron 629
	 3 connecs left for neuron 750
At layer encoder.encoder.layer.0.attention.self.query.weight, ============0/768================ neurons pruned at 0.9560195425536132%
	 4 connecs left for neuron 155
	 3 connecs left for neuron 201
	 2 connecs left for neuron 308
At layer encoder.encoder.layer.0.attention.self.key.weight, ============0/768================ neurons pruned at 0.9560195425536132%
	 2 connecs left for neuron 46
	 3 connecs left for neuron 53
	 2 connecs left for neuron 105
	 1 connecs left for neuron 121
	 1 connecs left for neuron 139
	 Layer encoder.encoder.layer.0.attention.self.value.weight: 145 pruned
	 Layer encoder.encoder.layer.0.attention.self.value.weight: 159 pruned
	 3 connecs left for neuron 168
	 2 connecs left for neuron 175
	 2 connecs left

In [64]:
per_layer_pruning_differences(11, layers, pruning_changes)

At layer encoder.encoder.layer.0.attention.self.query.weight, ============0/768================ neurons pruned at 0.914100765053316%
	 2 connecs left for neuron 308
At layer encoder.encoder.layer.0.attention.self.key.weight, ============0/768================ neurons pruned at 0.914100765053316%
	 3 connecs left for neuron 53
	 1 connecs left for neuron 121
	 1 connecs left for neuron 139
	 Layer encoder.encoder.layer.0.attention.self.value.weight: 145 pruned
	 1 connecs left for neuron 159
	 3 connecs left for neuron 168
	 4 connecs left for neuron 175
	 3 connecs left for neuron 202
	 4 connecs left for neuron 216
	 4 connecs left for neuron 283
	 4 connecs left for neuron 296
	 Layer encoder.encoder.layer.0.attention.self.value.weight: 308 pruned
	 3 connecs left for neuron 345
	 2 connecs left for neuron 353
	 2 connecs left for neuron 443
	 2 connecs left for neuron 545
	 2 connecs left for neuron 654
At layer encoder.encoder.layer.0.attention.self.value.weight, ============2/768==

In [ ]:
per_layer_pruning_differences(3, layers, pruning_changes)

In [ ]:
per_layer_pruning_differences(3, layers, pruning_changes)